# 01 — Com'è fatto un match: i quattro file

Ogni partita in `data/matches/{id}/` ha quattro file. Questo notebook li apre uno
per uno per capire cosa contengono, con che granularità e come si collegano fra
loro.

Il risultato è documentato in [`notes/dati/`](../notes/dati/), un file per tipo.
Qui resta il codice che ha prodotto quelle conclusioni.

In [1]:
import sys
sys.path.append("..")

import json
import numpy as np
import pandas as pd

from lib import data

MATCH_ID = 1886347
pd.set_option("display.width", 130)

## 1. `{id}_match.json` — i metadati

È il file da cui partire: dà le chiavi per interpretare tutto il resto
(chi è chi, quanto è grande il campo, dove iniziano e finiscono i tempi).

In [2]:
meta = data.load_match_meta(MATCH_ID)

def forma(v):
    if isinstance(v, dict):
        return "{" + ", ".join(sorted(v)[:6]) + ("…}" if len(v) > 6 else "}")
    if isinstance(v, list):
        return f"[{len(v)}] " + (forma(v[0]) if v else "")
    return type(v).__name__

for k in sorted(meta):
    print(f"{k:26s} {forma(meta[k])}")

away_team                  {acronym, id, name, short_name}
away_team_coach            NoneType
away_team_kit              {id, jersey_color, name, number_color, season, team_id}
away_team_playing_time     {minutes_otip, minutes_tip}
away_team_score            int
ball                       {trackable_object}
competition_edition        {competition, id, name, season}
competition_round          {id, name, potential_overtime, round_number}
date_time                  str
home_team                  {acronym, id, name, short_name}
home_team_coach            NoneType
home_team_kit              {id, jersey_color, name, number_color, season, team_id}
home_team_playing_time     {minutes_otip, minutes_tip}
home_team_score            int
home_team_side             [2] str
id                         int
match_periods              [2] {duration_frames, duration_minutes, end_frame, name, period, start_frame}
pitch_length               int
pitch_width                int
players                    [36]

In [3]:
print("campo:", meta["pitch_length"], "x", meta["pitch_width"], "m")
print("verso di attacco della squadra di casa, per tempo:", meta["home_team_side"])
print()
for p in meta["match_periods"]:
    print(f"  {p['name']}: frame {p['start_frame']}-{p['end_frame']} "
          f"({p['duration_minutes']} min)")

campo: 104 x 68 m
verso di attacco della squadra di casa, per tempo: ['right_to_left', 'left_to_right']

  period_1: frame 10-27790 (46.3 min)
  period_2: frame 27800-59060 (52.1 min)


`home_team_side` è la cosa da non dimenticare: **la squadra di casa cambia lato fra
primo e secondo tempo.** Qualunque metrica spaziale aggregata sui due tempi senza
normalizzare il verso è sbagliata.

`players` contiene le due rose. Serve per due mapping che userai ovunque:
`player_id -> team_id` e `player_id -> ruolo`.

In [4]:
squadra_di = {p["id"]: p["team_id"] for p in meta["players"]}
ruolo_di = {p["id"]: p["player_role"]["acronym"]
            for p in meta["players"] if p.get("player_role")}

print("giocatori nel file:", len(meta["players"]))
print("con ruolo assegnato:", len(ruolo_di), "(gli altri non sono scesi in campo)")
print()
print(pd.DataFrame(meta["players"])[["id", "last_name", "number", "team_id"]].head(4).to_string(index=False))

giocatori nel file: 36
con ruolo assegnato: 36 (gli altri non sono scesi in campo)

   id      last_name  number  team_id
38673 May Bartesaghi      10     4177
51713         Elliot      17     4177
50951        Brimmer      22     4177
50978        Timmins      19     1805


## 2. `{id}_tracking_extrapolated.jsonl` — le posizioni

Una riga JSON per frame, 10 fps. È il file grosso (~90 MB) e l'unico su Git LFS.

In [5]:
primo = next(iter(data.iter_tracking(MATCH_ID, limit=1)))
print("chiavi del frame:", list(primo))

esempio = [f for f in data.iter_tracking(MATCH_ID, limit=1500)][-1]
print()
print("ball_data: ", esempio["ball_data"])
print("possession:", esempio["possession"])
print("player_data[0]:", esempio["player_data"][0])
print()
print("image_corners_projection:", list(esempio["image_corners_projection"]))

chiavi del frame: ['frame', 'timestamp', 'period', 'ball_data', 'possession', 'image_corners_projection', 'player_data']

ball_data:  {'x': 40.51, 'y': 20.35, 'z': 0.19, 'is_detected': True}
possession: {'player_id': None, 'group': 'home team'}
player_data[0]: {'x': -27.87, 'y': 0.94, 'player_id': 51009, 'is_detected': False}

image_corners_projection: ['x_top_left', 'y_top_left', 'x_bottom_left', 'y_bottom_left', 'x_bottom_right', 'y_bottom_right', 'x_top_right', 'y_top_right']


Tre cose da notare:

- **anche la palla ha `is_detected`**, e ha una `z` (altezza);
- `possession` dà la squadra (`group`) e, quando identificato, il giocatore;
- `image_corners_projection` è la porzione di campo inquadrata, come quattro
  angoli — la via diretta per sapere cosa la telecamera stava vedendo.

Quanti frame ci sono davvero, e quanti sono utilizzabili?

In [6]:
tot = vuoti = senza_palla = 0
conteggio = {}

for f in data.iter_tracking(MATCH_ID):
    tot += 1
    n = len(f["player_data"])
    conteggio[n] = conteggio.get(n, 0) + 1
    if n == 0:
        vuoti += 1
    if not (f.get("ball_data") or {}).get("x"):
        senza_palla += 1

print(f"frame totali:          {tot:,}")
print(f"  senza giocatori:     {vuoti:,} ({100*vuoti/tot:.0f}%)")
print(f"  senza palla:         {senza_palla:,} ({100*senza_palla/tot:.0f}%)")
print()
print("giocatori per frame:", dict(sorted(conteggio.items(), key=lambda x: -x[1])))

frame totali:          59,061
  senza giocatori:     15,603 (26%)
  senza palla:         15,617 (26%)

giocatori per frame: {22: 43458, 0: 15603}


**Il tracking è tutto-o-niente**: un frame ha 22 giocatori oppure zero, mai valori
intermedi. Il 26% dei frame è vuoto — gioco fermo, replay, inquadrature che non
mostrano il campo.

Quindi i frame "utili" sono ~43.500 su ~59.000, cioè circa 72 minuti di gioco
osservabile su 98 di durata nominale.

## 3. `{id}_dynamic_events.csv` — gli eventi derivati

Non sono eventi annotati a mano come in StatsBomb: sono **derivati dal tracking
dai modelli di SkillCorner**. 322 colonne.

In [7]:
de = data.load_dynamic_events(MATCH_ID)
print("shape:", de.shape)
print()
print(de.event_type.value_counts().to_string())
print()
for t, g in de.groupby("event_type"):
    sub = sorted(g.event_subtype.dropna().unique())
    print(f"{t:20s} -> {sub if sub else '(nessun sottotipo)'}")

shape: (5115, 322)

event_type
passing_option        2573
player_possession      990
on_ball_engagement     957
off_ball_run           595

off_ball_run         -> ['behind', 'coming_short', 'cross_receiver', 'dropping_off', 'overlap', 'pulling_half_space', 'pulling_wide', 'run_ahead_of_the_ball', 'support', 'underlap']
on_ball_engagement   -> ['counter_press', 'other', 'pressing', 'pressure', 'recovery_press']
passing_option       -> (nessun sottotipo)
player_possession    -> (nessun sottotipo)


In [8]:
# Le 322 colonne non valgono per tutti i tipi: ognuno ne popola un sottoinsieme.
for t, g in de.groupby("event_type"):
    popolate = [c for c in de.columns if g[c].notna().mean() > 0.1]
    print(f"{t:20s} {len(popolate):3d} colonne popolate")

off_ball_run         115 colonne popolate
on_ball_engagement   131 colonne popolate
passing_option       143 colonne popolate
player_possession    205 colonne popolate


Il file contiene le **metriche modellate** di SkillCorner, non solo descrittori
geometrici: `xthreat`, `xpass_completion`, `xshot`, `xloss`, l'EPV,
`reception_difficulty`, `space_constraint`, `overall_pressure`.

Sono utili e sono una scorciatoia, ma vanno trattate come **output di modelli
chiusi di terze parti**: se ci costruisci sopra un risultato, quel risultato
eredita assunzioni che non puoi ispezionare.

In [9]:
modellate = ["xthreat", "xpass_completion", "xshot_player_possession_max",
             "xloss_player_possession_max", "possession_epv_total", "pass_epv_total"]
print(de[modellate].notna().sum().to_string())
print()
print(de[modellate].describe().loc[["mean", "min", "max"]].round(3).to_string())

xthreat                        3168
xpass_completion               3168
xshot_player_possession_max     956
xloss_player_possession_max     956
possession_epv_total            970
pass_epv_total                  820

      xthreat  xpass_completion  xshot_player_possession_max  xloss_player_possession_max  possession_epv_total  pass_epv_total
mean    0.011             0.821                        0.048                        0.293                 0.002           0.001
min     0.000             0.056                        0.000                        0.004                -0.089          -0.189
max     0.422             1.000                        0.938                        0.510                 0.202           0.146


### Il file dichiara la propria qualità

`fully_extrapolated` marca gli eventi costruiti interamente su posizioni stimate —
il collegamento diretto con l'esplorazione
[00](00-quanto-e-osservato.ipynb).

In [10]:
print(de.groupby("event_type").fully_extrapolated
        .agg(eventi="count", estrapolati="sum", quota="mean").round(3).to_string())
print()
for c in ["is_player_possession_start_matched", "is_previous_pass_matched",
          "is_pass_reception_matched"]:
    s = de[c]
    print(f"{c:38s} {int(s.sum()):5d} / {s.notna().sum():5d} allineati")

                    eventi estrapolati     quota
event_type                                      
off_ball_run             0           0       NaN
on_ball_engagement       0           0       NaN
passing_option        2573         357  0.138749
player_possession      990           4   0.00404

is_player_possession_start_matched      5075 /  5115 allineati
is_previous_pass_matched                1923 /  1946 allineati
is_pass_reception_matched               2428 /  2448 allineati


## 4. `{id}_phases_of_play.csv` — le fasi di gioco

Segmenta la partita in fasi. Una riga copre un intervallo di frame e descrive
**entrambe le squadre insieme**: cosa fa chi ha la palla e cosa fa chi non ce l'ha.

In [11]:
pop = data.load_phases_of_play(MATCH_ID)
print("shape:", pop.shape)
print()
print(pop[["frame_start", "frame_end", "duration", "team_in_possession_shortname",
           "team_in_possession_phase_type", "team_out_of_possession_phase_type"]]
      .head(5).to_string(index=False))

shape: (449, 44)

 frame_start  frame_end  duration team_in_possession_shortname team_in_possession_phase_type team_out_of_possession_phase_type
          28         89       6.1                    Newcastle                        create                      medium_block
          89        124       3.5                    Newcastle                        direct                  defending_direct
         189        232       4.3                  Auckland FC                      build_up                        high_block
         232        283       5.1                  Auckland FC                        create                      medium_block
         283        299       1.6                    Newcastle                       chaotic                           chaotic


In [12]:
coppie = (pop.groupby(["team_in_possession_phase_type",
                       "team_out_of_possession_phase_type"])
             .size().rename("fasi").reset_index()
             .sort_values("fasi", ascending=False))
coppie

,team_in_possession_phase_type,team_out_of_possession_phase_type,fasi
2,create,medium_block,156
4,finish,low_block,94
1,chaotic,chaotic,82
0,build_up,high_block,66
3,direct,defending_direct,25
6,set_play,defending_set_play,21
5,quick_break,defending_quick_break,4
7,transition,defending_transition,1


Le due tassonomie sono **speculari**: a ogni fase offensiva corrisponde
esattamente una fase difensiva. `build_up` ↔ `high_block`, `create` ↔
`medium_block`, `finish` ↔ `low_block`, e così via. Non sono due etichette
indipendenti, sono due nomi dello stesso momento visto dalle due parti.

In [13]:
print("durata fase: mediana %.1f s, max %.1f s" % (pop.duration.median(), pop.duration.max()))
print("tempo coperto dalle fasi: %.0f minuti" % (pop.duration.sum() / 60))
print()
# Larghezza e lunghezza della squadra sono già calcolate: compattezza gratis.
print(pop[["team_in_possession_width_start", "team_in_possession_length_start",
           "team_out_of_possession_width_start", "team_out_of_possession_length_start"]]
      .describe().loc[["mean", "min", "max"]].round(1).to_string())

durata fase: mediana 5.2 s, max 46.9 s
tempo coperto dalle fasi: 53 minuti

      team_in_possession_width_start  team_in_possession_length_start  team_out_of_possession_width_start  team_out_of_possession_length_start
mean                            42.7                             53.4                                35.6                                 51.0
min                             12.6                             16.5                                10.8                                  7.5
max                             66.4                             77.2                                62.0                                 80.3


`width` e `length` della squadra sono già calcolate a inizio e fine fase: la
compattezza di reparto è disponibile senza doverla derivare dal tracking.

### Il vocabolario delle zone

Le stesse etichette tornano in `dynamic_events` e `phases_of_play`, quindi vale la
pena impararle una volta sola.

In [14]:
print("channel (fasce verticali):", sorted(pop.channel_start.dropna().unique()))
print("third   (terzi di campo): ", sorted(pop.third_start.dropna().unique()))

channel (fasce verticali): ['center', 'half_space_left', 'half_space_right', 'wide_left', 'wide_right']
third   (terzi di campo):  ['attacking_third', 'defensive_third', 'middle_third']


## Come si collegano

Tutto si aggancia su **`frame`**, che è la chiave universale di una partita.

| Da | A | Come |
|---|---|---|
| dynamic_events | tracking | `frame_start` / `frame_end` → `frame` |
| phases_of_play | tracking | `frame_start` / `frame_end` → `frame` |
| dynamic_events | phases_of_play | `phase_index` |
| tracking | match.json | `player_id` → `players[].id` |
| dynamic_events | match.json | `player_id`, `team_id` |

Ed è anche il motivo per cui animare una fase di gioco è immediato: prendi
`frame_start`/`frame_end` da un evento o da una fase, e filtri il tracking.

La documentazione per file sta in [`notes/dati/`](../notes/dati/).